# DELCODE DMN Split Repro Check (GAAE + GEC)

This notebook reproduces the split generation logic from `DATA/DELCODE/__v4__/dmn_only_schaefer/_artifacts/generate_dmn_splits.py` using the same seed and fractions, then compares the recomputed splits with the saved split CSVs under `DATA/DELCODE/__v4__/metadata`.

It also reports statistics for both split families:
- subject counts per split
- diagnosis and converter distributions
- scan-count summaries
- overlap/disjointness checks

In [1]:
from collections import Counter, defaultdict
from math import ceil
from pathlib import Path
import random

import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
DMN_DIR = Path('/mnt/e/fyassine/ad-early-detection/DATA/DELCODE/__v4__/dmn_only_schaefer')
METADATA_DIR = Path('/mnt/e/fyassine/ad-early-detection/DATA/DELCODE/__v4__/metadata')
COHORTS_CSV = METADATA_DIR / 'cohorts.csv'
SUFFIX = '_dmn_correlation_matrix_z_transformed.npz'
EXCLUDED_DIAGNOSES = {'scd'}
SEED = 42
TRAIN_FRAC = 0.75
VAL_FRAC = 0.15
TEST_FRAC = 1.0 - TRAIN_FRAC - VAL_FRAC

GAAE_DIR = METADATA_DIR / 'splits_gaae'
GEC_DIR = METADATA_DIR / 'splits_gec'

print(f'DMN dir: {DMN_DIR}')
print(f'Metadata dir: {METADATA_DIR}')
print(f'Fractions: train={TRAIN_FRAC:.0%}, val={VAL_FRAC:.0%}, test={TEST_FRAC:.0%}, seed={SEED}')

DMN dir: /mnt/e/fyassine/ad-early-detection/DATA/DELCODE/__v4__/dmn_only_schaefer
Metadata dir: /mnt/e/fyassine/ad-early-detection/DATA/DELCODE/__v4__/metadata
Fractions: train=75%, val=15%, test=10%, seed=42


In [3]:
def extract_subject_id(filename: str) -> str:
    return filename.split('_')[0].replace('sub-', '')


def _can_stratify(labels, test_size) -> bool:
    n_samples = len(labels)
    if n_samples < 2:
        return False

    counts = Counter(labels)
    if min(counts.values()) < 2:
        return False

    if isinstance(test_size, float):
        n_test = ceil(n_samples * test_size)
    else:
        n_test = int(test_size)
    n_train = n_samples - n_test

    n_classes = len(counts)
    if n_test < n_classes or n_train < n_classes:
        return False
    return True


def _safe_train_test_split(subjects, labels, test_size, seed, split_name):
    stratify = labels if _can_stratify(labels, test_size) else None
    if stratify is None:
        print(f'WARNING: {split_name} cannot be stratified; using random split with seed={seed}.')

    train_subjects, test_subjects = train_test_split(
        subjects,
        test_size=test_size,
        random_state=seed,
        stratify=stratify,
    )
    label_map = dict(zip(subjects, labels))
    train_labels = [label_map[s] for s in train_subjects]
    test_labels = [label_map[s] for s in test_subjects]
    return train_subjects, test_subjects, train_labels, test_labels


def _collect_subject_to_files(data_dir: Path):
    z_files = sorted(data_dir.glob(f'*{SUFFIX}'))
    if not z_files:
        raise FileNotFoundError(f'No files matching *{SUFFIX} in {data_dir}')

    subject_to_files = defaultdict(list)
    for file_path in z_files:
        sid = extract_subject_id(file_path.name)
        subject_to_files[sid].append(file_path.name)
    return subject_to_files


def _baseline_diagnosis_map(cohorts_csv: Path):
    cohorts_df = pd.read_csv(cohorts_csv)
    required_cols = {'Pseudonym', 'diagnosis', 'sex', 'brthdat'}
    missing = required_cols - set(cohorts_df.columns)
    if missing:
        raise ValueError(f'cohorts.csv missing required columns: {sorted(missing)}')

    diagnosis_map = {}
    sex_map = {}
    age_map = {}

    for _, row in cohorts_df.iterrows():
        sid = str(row['Pseudonym'])
        if sid in diagnosis_map:
            continue

        diagnosis_map[sid] = str(row['diagnosis']).strip().lower()
        sex_map[sid] = str(row['sex']).strip().lower()

        brthdat = str(row['brthdat']).strip()
        if pd.notna(brthdat) and brthdat and brthdat != 'nan':
            try:
                birth_year = int(brthdat.split('-')[0])
                age_map[sid] = max(18, min(2023 - birth_year, 120))
            except (ValueError, IndexError):
                age_map[sid] = 50
        else:
            age_map[sid] = 50

    return diagnosis_map, sex_map, age_map


def _build_subject_table(subject_to_files, subject_to_diagnosis, subject_to_sex, subject_to_age, excluded_diagnoses):
    records = []
    for sid in sorted(subject_to_files.keys()):
        diagnosis = subject_to_diagnosis.get(sid)
        if diagnosis is None or diagnosis in excluded_diagnoses:
            continue

        records.append({
            'Repseudonym': sid,
            'diagnosis': diagnosis,
            'converter_status': 1 if diagnosis == 'converter' else 0,
            'sex': subject_to_sex.get(sid, 'unknown'),
            'age': subject_to_age.get(sid, 50),
            'n_scans': len(subject_to_files[sid]),
        })

    if not records:
        raise ValueError('No subjects available after metadata join and exclusions.')

    return pd.DataFrame(records).sort_values('Repseudonym').reset_index(drop=True)


def _make_gaae_splits(subject_df, train_frac, val_frac, seed):
    subject_ids = subject_df['Repseudonym'].tolist()
    diagnosis_labels = subject_df['diagnosis'].tolist()

    test_frac = 1.0 - train_frac - val_frac
    val_test_frac = val_frac + test_frac

    train_ids, valtest_ids, _, valtest_labels = _safe_train_test_split(
        subject_ids,
        diagnosis_labels,
        test_size=val_test_frac,
        seed=seed,
        split_name='GAAE train vs (val+test)',
    )

    relative_test_frac = test_frac / val_test_frac
    val_ids, test_ids, _, _ = _safe_train_test_split(
        valtest_ids,
        valtest_labels,
        test_size=relative_test_frac,
        seed=seed,
        split_name='GAAE val vs test',
    )

    return {
        'train': sorted(train_ids),
        'val': sorted(val_ids),
        'test': sorted(test_ids),
    }


def _balanced_class_count_split(total_per_class, train_frac, val_frac):
    n_train = int(total_per_class * train_frac)
    n_val = int(total_per_class * val_frac)
    n_test = total_per_class - n_train - n_val
    return n_train, n_val, n_test


def _make_gec_splits(subject_df, train_frac, val_frac, seed):
    converters = subject_df[subject_df['converter_status'] == 1]['Repseudonym'].tolist()
    non_converters = subject_df[subject_df['converter_status'] == 0]['Repseudonym'].tolist()

    target_per_class = min(len(converters), len(non_converters))
    rng = random.Random(seed)

    selected_converters = sorted(rng.sample(converters, target_per_class))
    selected_non_converters = sorted(rng.sample(non_converters, target_per_class))

    rng.shuffle(selected_converters)
    rng.shuffle(selected_non_converters)

    n_train, n_val, n_test = _balanced_class_count_split(target_per_class, train_frac, val_frac)

    train_ids = selected_converters[:n_train] + selected_non_converters[:n_train]
    val_ids = selected_converters[n_train:n_train + n_val] + selected_non_converters[n_train:n_train + n_val]
    test_ids = selected_converters[n_train + n_val:] + selected_non_converters[n_train + n_val:]

    return {
        'train': sorted(train_ids),
        'val': sorted(val_ids),
        'test': sorted(test_ids),
    }


def _load_split_dir(split_dir: Path):
    return {name: pd.read_csv(split_dir / f'{name}.csv') for name in ['train', 'val', 'test']}


def _split_subject_sets(split_frames):
    return {name: set(df['Repseudonym'].astype(str)) for name, df in split_frames.items()}


def _print_overlap_report(split_sets, title):
    tv = len(split_sets['train'] & split_sets['val'])
    tt = len(split_sets['train'] & split_sets['test'])
    vt = len(split_sets['val'] & split_sets['test'])
    print(f'{title} overlaps -> train/val={tv}, train/test={tt}, val/test={vt}')


def _print_stats(split_frames, title):
    print(f'\n===== {title} =====')
    for split_name in ['train', 'val', 'test']:
        df = split_frames[split_name]   
        print(f'{split_name}: {len(df)} subjects, {df["n_scans"].sum() if "n_scans" in df.columns else 0} scans')

    all_df = pd.concat([split_frames[s].assign(split=s) for s in ['train', 'val', 'test']], ignore_index=True)

    if 'diagnosis' in all_df.columns:
        print('\nDiagnosis distribution by split:')
        print(pd.crosstab(all_df['diagnosis'], all_df['split']).sort_index())

    if 'converter_status' in all_df.columns:
        print('\nConverter distribution by split (0=non-converter, 1=converter):')
        print(pd.crosstab(all_df['converter_status'], all_df['split']).sort_index())

    if 'n_scans' in all_df.columns and 'diagnosis' in all_df.columns:
        print('\nScan totals by diagnosis cohort and split:')
        print(all_df.groupby(['diagnosis', 'split'])['n_scans'].sum().unstack(fill_value=0).sort_index())

    if 'n_scans' in all_df.columns and 'converter_status' in all_df.columns:
        print('\nScan totals by converter cohort and split (0=non-converter, 1=converter):')
        print(all_df.groupby(['converter_status', 'split'])['n_scans'].sum().unstack(fill_value=0).sort_index())

    if 'n_scans' in all_df.columns:
        print('\nScan-count summary by split:')
        print(all_df.groupby('split')['n_scans'].agg(['count', 'sum', 'mean', 'median', 'min', 'max']).round(3))

In [6]:
subject_to_files = _collect_subject_to_files(DMN_DIR)
subject_to_diagnosis, subject_to_sex, subject_to_age = _baseline_diagnosis_map(COHORTS_CSV)
subject_df = _build_subject_table(
    subject_to_files=subject_to_files,
    subject_to_diagnosis=subject_to_diagnosis,
    subject_to_sex=subject_to_sex,
    subject_to_age=subject_to_age,
    excluded_diagnoses=EXCLUDED_DIAGNOSES,
)

print(f'Subjects with DMN files: {len(subject_to_files)}')
print(f'Subjects used for splitting: {len(subject_df)}')
print('Diagnosis totals (eligible subjects):')
print(subject_df['diagnosis'].value_counts().sort_index())

Subjects with DMN files: 840
Subjects used for splitting: 460
Diagnosis totals (eligible subjects):
diagnosis
ad           103
converter     58
healthy      200
mci           99
Name: count, dtype: int64


In [5]:
expected_gaae_sets = _make_gaae_splits(subject_df, TRAIN_FRAC, VAL_FRAC, SEED)
expected_gec_sets = _make_gec_splits(subject_df, TRAIN_FRAC, VAL_FRAC, SEED)

saved_gaae = _load_split_dir(GAAE_DIR)
saved_gec = _load_split_dir(GEC_DIR)

saved_gaae_sets = _split_subject_sets(saved_gaae)
saved_gec_sets = _split_subject_sets(saved_gec)

def compare_sets(expected, actual, title):
    print(f'\n===== Repro Check: {title} =====')
    all_match = True
    for split in ['train', 'val', 'test']:
        e = set(expected[split])
        a = set(actual[split])
        only_expected = sorted(e - a)
        only_actual = sorted(a - e)
        match = (e == a)
        all_match = all_match and match
        print(f'{split}: match={match}, expected={len(e)}, actual={len(a)}')
        if not match:
            print(f'  only_expected (first 10): {only_expected[:10]}')
            print(f'  only_actual   (first 10): {only_actual[:10]}')

    print(f'Overall exact match: {all_match}')

compare_sets(expected_gaae_sets, saved_gaae_sets, 'GAAE')
compare_sets(expected_gec_sets, saved_gec_sets, 'GEC')


===== Repro Check: GAAE =====
train: match=True, expected=345, actual=345
val: match=True, expected=69, actual=69
test: match=True, expected=46, actual=46
Overall exact match: True

===== Repro Check: GEC =====
train: match=True, expected=86, actual=86
val: match=True, expected=16, actual=16
test: match=True, expected=14, actual=14
Overall exact match: True


In [8]:
_print_overlap_report(saved_gaae_sets, 'Saved GAAE')
_print_overlap_report(saved_gec_sets, 'Saved GEC')

_print_stats(saved_gaae, 'Saved GAAE split stats')
_print_stats(saved_gec, 'Saved GEC split stats')

Saved GAAE overlaps -> train/val=0, train/test=0, val/test=0
Saved GEC overlaps -> train/val=0, train/test=0, val/test=0

===== Saved GAAE split stats =====
train: 345 subjects, 447 scans
val: 69 subjects, 92 scans
test: 46 subjects, 59 scans

Diagnosis distribution by split:
split      test  train  val
diagnosis                  
ad           10     77   16
converter     6     44    8
healthy      20    150   30
mci          10     74   15

Scan totals by diagnosis cohort and split:
split      test  train  val
diagnosis                  
ad           10     77   16
converter    19    146   31
healthy      20    150   30
mci          10     74   15

Scan-count summary by split:
       count  sum   mean  median  min  max
split                                     
test      46   59  1.283     1.0    1    5
train    345  447  1.296     1.0    1    6
val       69   92  1.333     1.0    1    5

===== Saved GEC split stats =====
train: 86 subjects, 187 scans
val: 16 subjects, 34 scans
test: 